<a href="https://colab.research.google.com/github/jonasknoll57/Bachelorarbeit_Demand-AD/blob/main/V17c_Pair_Comparison.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# V17c — Cross-City Pair Comparison

**Zweck:** Robustheits-/Pair-Comparison-Phase.
Prueft ob das Transfer-Ergebnis aus V17b (HD -> MA) ein Sonderfall war oder
sich ueber mehrere Targets und Source-Paare bestaetigt.

**Leitfragen:**
1. Ist Strict Zero-Shot bei aehnlichen Sources (+) systematisch besser als bei unaehnlicheren (-)?
2. Bringt Fine-Tune 5% das Modell konsistent naeher an das Oracle?
3. Ist Transfer besser als Target-only 5%?
4. Bleibt das Ergebnisbild ueber mehrere Targets hinweg plausibel?

**Targets:** Mannheim, Freiburg, Braunschweig
**Experimente pro Target:** Oracle 100%, Target-only 5%, ZS Strict (+), ZS Strict (-), FT 5% (+), FT 5% (-)
**Modell:** LSTM Forecaster (directional scoring + Z-Norm)
**Anomalietypen:** point_spike, point_drop, collective (keine contextuals)


In [1]:
# 0a — Colab Setup
from google.colab import drive
drive.mount('/content/drive', force_remount=False)


Mounted at /content/drive


In [2]:
# 0b — Imports
import os, math, json, random, warnings, time, re, glob, copy, pickle
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

import numpy as np
import pandas as pd

from scipy.stats import poisson
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve,
    classification_report
)
from numpy.lib.stride_tricks import sliding_window_view

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

warnings.filterwarnings("ignore")
print(f"TF {tf.__version__}, GPU: {tf.config.list_physical_devices('GPU')}")


TF 2.19.0, GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
# 0c — V17c Config
DATA_BASE    = '/content/drive/MyDrive/BA_Colab/data'
CLEANED_BASE = '/content/drive/MyDrive/BA_Colab/cleaned'

RUN_NAME    = 'v17c_pair_comparison'
RESULTS_DIR = f'{DATA_BASE}/{RUN_NAME}'
MODELS_DIR  = f'{RESULTS_DIR}/models'
V17_MODELS  = f'{DATA_BASE}/v17_ad_improvements/models'
V17B_MODELS = f'{DATA_BASE}/v17b_transfer_sanity/models'
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

geo_path           = f'{CLEANED_BASE}/geo_information/geo_information.parquet'
station_names_path = f'{DATA_BASE}/station_names/station_names.parquet'

CITY_DEMAND = {
    "Mannheim":       f'{CLEANED_BASE}/demand/Mannheim/demand_cleaned.parquet',
    "Heidelberg":     f'{CLEANED_BASE}/demand/Heidelberg/demand_cleaned.parquet',
    "Marburg":        f'{CLEANED_BASE}/demand/Marburg/demand_cleaned.parquet',
    "Freiburg":       f'{CLEANED_BASE}/demand/Freiburg/demand_cleaned.parquet',
    "Ludwigshafen":   f'{CLEANED_BASE}/demand/Ludwigshafen/demand_cleaned.parquet',
    "Braunschweig":   f'{CLEANED_BASE}/demand/Braunschweig/demand_cleaned.parquet',
    "Kaiserslautern": f'{CLEANED_BASE}/demand/Kaiserslautern/demand_cleaned.parquet',
}

EXPERIMENT_PAIRS = {
    "Mannheim": [
        ("Heidelberg", "+"),
        ("Marburg",    "-"),
    ],
    "Freiburg": [
        ("Heidelberg",   "+"),
        ("Ludwigshafen", "-"),
    ],
    "Braunschweig": [
        ("Mannheim",       "+"),
        ("Kaiserslautern", "-"),
    ],
}

@dataclass
class V17cConfig:
    aggregation_minutes: int = 60
    train_ratio: float = 0.67
    val_ratio: float = 0.82
    min_events_per_day: float = 3.0
    rolling_days: int = 56
    min_context_obs: int = 20
    ae_window_size: int = 24
    ae_latent_dim: int = 32
    ae_layers: int = 2
    ae_dropout: float = 0.10
    ae_epochs: int = 50
    ae_batch_size: int = 2048
    ae_lr: float = 1e-3
    ae_early_stop: int = 8
    eval_batch_size: int = 2048
    low_demand_q: float = 0.33
    high_demand_q: float = 0.67
    require_contiguous: bool = True
    use_bidirectional: bool = False
    ae_features: List[str] = field(default_factory=lambda: [
        "n_lends", "n_returns",
        "hour_sin", "hour_cos", "dow_sin", "dow_cos",
        "month_sin", "month_cos", "is_weekend"
    ])
    ae_score_features: List[str] = field(default_factory=lambda: ["n_lends", "n_returns"])
    injection_rate: float = 0.015
    injection_seed: int = 42
    finetune_ratio: float = 0.05

cfg = V17cConfig()
print(cfg)
print(f"\nResults: {RESULTS_DIR}")
print(f"V17 Models: {V17_MODELS}")
print(f"V17b Models: {V17B_MODELS}")

print("\n--- Demand-Dateien ---")
for city, path in CITY_DEMAND.items():
    exists = os.path.exists(path)
    print(f"  {city:18s}: {'OK' if exists else 'FEHLT!'} -- {path}")


V17cConfig(aggregation_minutes=60, train_ratio=0.67, val_ratio=0.82, min_events_per_day=3.0, rolling_days=56, min_context_obs=20, ae_window_size=24, ae_latent_dim=32, ae_layers=2, ae_dropout=0.1, ae_epochs=50, ae_batch_size=2048, ae_lr=0.001, ae_early_stop=8, eval_batch_size=2048, low_demand_q=0.33, high_demand_q=0.67, require_contiguous=True, use_bidirectional=False, ae_features=['n_lends', 'n_returns', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_weekend'], ae_score_features=['n_lends', 'n_returns'], injection_rate=0.015, injection_seed=42, finetune_ratio=0.05)

Results: /content/drive/MyDrive/BA_Colab/data/v17c_pair_comparison
V17 Models: /content/drive/MyDrive/BA_Colab/data/v17_ad_improvements/models
V17b Models: /content/drive/MyDrive/BA_Colab/data/v17b_transfer_sanity/models

--- Demand-Dateien ---
  Mannheim          : OK -- /content/drive/MyDrive/BA_Colab/cleaned/demand/Mannheim/demand_cleaned.parquet
  Heidelberg        : OK -- /content/drive/MyD

---
## Shared Functions (aus V17b uebernommen, unveraendert)


In [4]:
# 1 — Aggregation
def aggregate_station_level(df: pd.DataFrame, minutes: int = 60,
                            add_relative: bool = False) -> pd.DataFrame:
    out = df.copy()
    freq = f"{minutes}min"
    out["time_bin"] = out["timestamp"].dt.floor(freq)
    agg = (
        out.groupby(
            ["station_id", "station_name_id", "station_name", "location_id", "time_bin"],
            as_index=False
        )
        .agg({"n_lends": "sum", "n_returns": "sum", "latitude": "first", "longitude": "first"})
        .rename(columns={"time_bin": "hour_ts"})
    )
    agg["total_demand"] = agg["n_lends"] + agg["n_returns"]
    agg["net_flow"] = agg["n_returns"] - agg["n_lends"]
    agg["abs_net_flow"] = agg["net_flow"].abs()
    agg["hour"] = agg["hour_ts"].dt.hour
    agg["dow"] = agg["hour_ts"].dt.dayofweek
    agg["month"] = agg["hour_ts"].dt.month
    agg["is_weekend"] = agg["dow"].isin([5, 6]).astype(int)
    agg["hour_sin"] = np.sin(2 * np.pi * agg["hour"] / 24)
    agg["hour_cos"] = np.cos(2 * np.pi * agg["hour"] / 24)
    agg["dow_sin"]  = np.sin(2 * np.pi * agg["dow"] / 7)
    agg["dow_cos"]  = np.cos(2 * np.pi * agg["dow"] / 7)
    agg["month_sin"] = np.sin(2 * np.pi * (agg["month"] - 1) / 12)
    agg["month_cos"] = np.cos(2 * np.pi * (agg["month"] - 1) / 12)
    agg = agg.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    return agg


In [5]:
# 2 — Gap-Fill
def fill_missing_time_bins(x: pd.DataFrame, minutes: int = 60) -> pd.DataFrame:
    freq = f"{minutes}min"
    parts = []
    x = x.copy().sort_values(["station_id", "hour_ts"])
    x = (
        x.groupby(["station_id", "hour_ts"], as_index=False)
         .agg({
             "station_name_id": "first", "station_name": "first",
             "location_id": "first", "latitude": "first",
             "longitude": "first", "n_lends": "sum", "n_returns": "sum",
         })
    )
    key_cols = ["station_id", "station_name_id", "station_name",
                "location_id", "latitude", "longitude"]
    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").copy()
        full_idx = pd.date_range(g["hour_ts"].min(), g["hour_ts"].max(), freq=freq)
        g = g.set_index("hour_ts").reindex(full_idx)
        g.index.name = "hour_ts"
        g["n_lends"]   = g["n_lends"].fillna(0).astype(int)
        g["n_returns"] = g["n_returns"].fillna(0).astype(int)
        for col in key_cols:
            g[col] = g[col].ffill().bfill()
        parts.append(g.reset_index())
    result = pd.concat(parts, ignore_index=True)
    result["total_demand"] = result["n_lends"] + result["n_returns"]
    result["net_flow"]     = result["n_returns"] - result["n_lends"]
    result["abs_net_flow"] = result["net_flow"].abs()
    result["hour"] = result["hour_ts"].dt.hour
    result["dow"]  = result["hour_ts"].dt.dayofweek
    result["month"] = result["hour_ts"].dt.month
    result["is_weekend"] = result["dow"].isin([5, 6]).astype(int)
    result["hour_sin"]  = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"]  = np.cos(2 * np.pi * result["hour"] / 24)
    result["dow_sin"]   = np.sin(2 * np.pi * result["dow"] / 7)
    result["dow_cos"]   = np.cos(2 * np.pi * result["dow"] / 7)
    result["month_sin"] = np.sin(2 * np.pi * (result["month"] - 1) / 12)
    result["month_cos"] = np.cos(2 * np.pi * (result["month"] - 1) / 12)
    return result.sort_values(["station_id", "hour_ts"]).reset_index(drop=True)


In [6]:
# 3 — Station-Filter, Demand-Regime, Train/Val/Test Split
def prepare_stations_and_splits(df: pd.DataFrame, cfg, city_name=""):
    n_days = (df["hour_ts"].max() - df["hour_ts"].min()).days + 1
    min_events = int(n_days * cfg.min_events_per_day)
    station_volume = df.groupby("station_id")["total_demand"].sum()
    active_ids = station_volume[station_volume >= min_events].index.tolist()
    df = df[df["station_id"].isin(active_ids)].copy()
    station_profile = (
        df.groupby(["station_id", "station_name"], as_index=False)
          .agg(
              avg_total_demand_h=("total_demand", "mean"),
              avg_lends_h=("n_lends", "mean"),
              avg_returns_h=("n_returns", "mean"),
              latitude=("latitude", "first"),
              longitude=("longitude", "first")
          )
    )
    q1 = station_profile["avg_total_demand_h"].quantile(cfg.low_demand_q)
    q2 = station_profile["avg_total_demand_h"].quantile(cfg.high_demand_q)
    station_profile["demand_regime"] = station_profile["avg_total_demand_h"].apply(
        lambda x: "low" if x <= q1 else ("mid" if x <= q2 else "high")
    )
    df = df.merge(
        station_profile[["station_id", "demand_regime", "avg_total_demand_h"]],
        on="station_id", how="left"
    )
    t_min = df["hour_ts"].min()
    t_max = df["hour_ts"].max()
    total_h = (t_max - t_min).total_seconds() / 3600
    train_end = t_min + pd.Timedelta(hours=int(total_h * cfg.train_ratio))
    val_end   = t_min + pd.Timedelta(hours=int(total_h * cfg.val_ratio))
    cn = f" ({city_name})" if city_name else ""
    print(f"  Aktive Stationen{cn}: {df['station_id'].nunique()}")
    print(f"  Regime: {station_profile['demand_regime'].value_counts().to_dict()}")
    print(f"  Zeitraum: {t_min.date()} bis {t_max.date()} ({n_days} Tage)")
    print(f"  Split: Train < {train_end.date()}, Val < {val_end.date()}, Test ab {val_end.date()}")
    return df, station_profile, train_end, val_end


In [7]:
# 4 — Rolling Context z-Scores + Poisson + ECDF + Labels
TARGETS = ["n_lends", "n_returns", "net_flow", "total_demand"]
COUNT_TARGETS = ["n_lends", "n_returns", "total_demand"]

def add_context_keys(x):
    x = x.copy()
    x["ctx_hour"] = x["hour_ts"].dt.hour
    x["ctx_dow"]  = x["hour_ts"].dt.dayofweek
    return x

def rolling_context_scores_vectorized(full_df, target, rolling_days=56, min_obs=20):
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    mu_col = f"{target}_mu_ctx_roll"
    sd_col = f"{target}_sd_ctx_roll"
    score_col = f"{target}_z_ctx_roll"
    n_slots = max(rolling_days // 7, 4)
    main_window = n_slots
    main_minp = min(min_obs, main_window)
    grouped = x.groupby(["station_id", "ctx_hour", "ctx_dow"])[target]
    rolling_mean = grouped.transform(lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).mean())
    rolling_std = grouped.transform(lambda s: s.shift(1).rolling(window=main_window, min_periods=main_minp).std(ddof=0))
    fb1_window = n_slots * 2
    fb1_minp = min(min_obs, fb1_window)
    grouped_sh = x.groupby(["station_id", "ctx_hour"])[target]
    fb1_mean = grouped_sh.transform(lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).mean())
    fb1_std = grouped_sh.transform(lambda s: s.shift(1).rolling(window=fb1_window, min_periods=fb1_minp).std(ddof=0))
    fb2_window = n_slots * 4
    fb2_minp = min(min_obs, fb2_window)
    grouped_s = x.groupby(["station_id"])[target]
    fb2_mean = grouped_s.transform(lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).mean())
    fb2_std = grouped_s.transform(lambda s: s.shift(1).rolling(window=fb2_window, min_periods=fb2_minp).std(ddof=0))
    mu = rolling_mean.copy()
    sd = rolling_std.copy()
    mask1 = mu.isna()
    mu = mu.where(~mask1, fb1_mean)
    sd = sd.where(~mask1, fb1_std)
    mask2 = mu.isna()
    mu = mu.where(~mask2, fb2_mean)
    sd = sd.where(~mask2, fb2_std)
    sd = sd.clip(lower=1e-6)
    z = (x[target] - mu) / sd
    x[mu_col] = mu
    x[sd_col] = sd
    x[score_col] = z
    return x

def add_rolling_poisson_scores_vectorized(full_df, target, rolling_days=56, min_obs=20):
    x = full_df.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    lam_col = f"{target}_lambda_ctx_roll"
    score_col = f"{target}_poisson_upper_score"
    score_low_col = f"{target}_poisson_lower_score"
    mu_col = f"{target}_mu_ctx_roll"
    if mu_col not in x.columns:
        raise ValueError(f"{mu_col} muss zuerst berechnet werden")
    x[lam_col] = x[mu_col].clip(lower=1e-6)
    vals = x[target].values
    lams = x[lam_col].values
    with np.errstate(divide='ignore', invalid='ignore'):
        tail_p = poisson.sf(vals.astype(float) - 1, lams.astype(float))
        score = -np.log10(np.clip(tail_p, 1e-12, None))
        lower_p = poisson.cdf(vals.astype(float), lams.astype(float))
        score_low = -np.log10(np.clip(lower_p, 1e-12, None))
    mask_nan = np.isnan(lams)
    score[mask_nan] = np.nan
    score_low[mask_nan] = np.nan
    x[score_col] = score
    x[score_low_col] = score_low
    return x

def percentile_score(train_vals, values):
    train_vals = np.asarray(train_vals, dtype=float)
    values = np.asarray(values, dtype=float)
    train_vals = train_vals[np.isfinite(train_vals)]
    if len(train_vals) == 0:
        return np.full(len(values), np.nan, dtype=float)
    train_sorted = np.sort(train_vals)
    ranks = np.searchsorted(train_sorted, values, side="right")
    return ranks / len(train_sorted)

def calibrate_scores_by_station(full_df, train_mask, raw_score_cols):
    x = full_df.copy()
    for col in raw_score_cols:
        if col not in x.columns:
            continue
        pct_col = f"{col}_pct_station"
        x[pct_col] = np.nan
        for sid, grp in x.groupby("station_id", sort=False):
            idx_all = grp.index
            idx_train = grp.index[train_mask.loc[idx_all]]
            train_vals = x.loc[idx_train, col].to_numpy(dtype=float) if len(idx_train) > 0 else np.array([])
            vals = x.loc[idx_all, col].to_numpy(dtype=float)
            x.loc[idx_all, pct_col] = percentile_score(train_vals, vals)
    return x

def build_eval_labels_calibrated(x):
    y = x.copy()
    y["label_eval"] = "normal"
    z_up = "total_demand_score_upper_pct_station"
    z_low = "total_demand_score_lower_pct_station"
    p_up = "total_demand_poisson_upper_score_pct_station"
    p_low = "total_demand_poisson_lower_score_pct_station"
    required = [z_up, z_low, p_up, p_low]
    missing = [c for c in required if c not in y.columns]
    if missing:
        raise ValueError(f"Fehlende Spalten: {missing}")
    td = y["total_demand"].fillna(0)
    mu = y["total_demand_mu_ctx_roll"]
    abs_min_high = np.maximum(5, np.ceil(mu + 2)).fillna(5)
    abs_min_low_ref = np.maximum(3, np.ceil(mu)).fillna(3)
    high_strong = ((y[z_up] >= 0.995) | (y[p_up] >= 0.995)) & (td >= abs_min_high)
    high_consensus = ((y[z_up] >= 0.99) & (y[p_up] >= 0.99)) & (td >= abs_min_high)
    cond_high = high_strong | high_consensus
    low_possible = mu >= abs_min_low_ref
    low_strong = ((y[z_low] >= 0.999) | (y[p_low] >= 0.999)) & low_possible
    low_consensus = ((y[z_low] >= 0.995) & (y[p_low] >= 0.995)) & low_possible
    cond_low = (low_strong | low_consensus) & ~cond_high
    grau_high = ((y[z_up] >= 0.99) | (y[p_up] >= 0.99)) & ~cond_high & ~cond_low
    grau_low = ((y[z_low] >= 0.99) | (y[p_low] >= 0.99)) & ~cond_high & ~cond_low & ~grau_high
    y.loc[grau_high, "label_eval"] = "grauzone_high"
    y.loc[grau_low, "label_eval"] = "grauzone_low"
    y.loc[cond_high, "label_eval"] = "anomal_high"
    y.loc[cond_low, "label_eval"] = "anomal_low"
    return y

def run_statistical_pipeline(df, cfg, train_end):
    print("  [1/5] Kontext-Keys...")
    df = add_context_keys(df)
    print("  [2/5] Rolling z-Scores...")
    for tgt in TARGETS:
        df = rolling_context_scores_vectorized(df, target=tgt, rolling_days=cfg.rolling_days, min_obs=cfg.min_context_obs)
    for tgt in TARGETS:
        zc = f"{tgt}_z_ctx_roll"
        df[f"{tgt}_score_upper"] = df[zc]
        df[f"{tgt}_score_lower"] = -df[zc]
    print("  [3/5] Poisson-Tail-Scores...")
    for tgt in COUNT_TARGETS:
        df = add_rolling_poisson_scores_vectorized(df, target=tgt, rolling_days=cfg.rolling_days, min_obs=cfg.min_context_obs)
    print("  [4/5] ECDF-Kalibrierung...")
    raw_score_cols = []
    for tgt in TARGETS:
        raw_score_cols += [f"{tgt}_score_upper", f"{tgt}_score_lower"]
    for tgt in COUNT_TARGETS:
        raw_score_cols += [f"{tgt}_poisson_upper_score", f"{tgt}_poisson_lower_score"]
    train_mask = df["hour_ts"] < train_end
    df = calibrate_scores_by_station(df, train_mask, raw_score_cols)
    print("  [5/5] Labels...")
    df = build_eval_labels_calibrated(df)
    anomaly_rate = df["label_eval"].isin(["anomal_high", "anomal_low"]).mean()
    print(f"  Label-Verteilung:")
    print(f"    {df['label_eval'].value_counts().to_dict()}")
    print(f"    Anomalie-Rate: {anomaly_rate:.4f} ({anomaly_rate*100:.2f}%)")
    return df


In [8]:
# 5 — Sequenzbuilder (Window-Level Labels)
def make_sequences_with_window_labels(
    x, feature_cols, window_size,
    synth_label_col="synth_label", synth_type_col="synth_type",
    require_contiguous=True, agg_minutes=60
):
    X_parts, meta_parts = [], []
    expected_ns = pd.Timedelta(minutes=agg_minutes).value
    for sid, g in x.groupby("station_id", sort=False):
        g = g.sort_values("hour_ts").reset_index(drop=True)
        n = len(g)
        if n < window_size:
            continue
        vals = g[feature_cols].to_numpy(dtype=np.float32)
        win = sliding_window_view(vals, window_shape=window_size, axis=0)
        win = np.moveaxis(win, -1, 1)
        valid_mask = np.ones(n - window_size + 1, dtype=bool)
        if require_contiguous:
            ts_int = pd.to_datetime(g["hour_ts"]).astype("int64").to_numpy()
            diffs = np.diff(ts_int)
            step_ok = (diffs == expected_ns).astype(np.int8)
            if window_size > 1:
                ok_sums = np.convolve(step_ok, np.ones(window_size-1, dtype=np.int32), mode="valid")
                valid_mask = (ok_sums == (window_size - 1))
        if not valid_mask.any():
            continue
        end_indices = np.arange(window_size - 1, n)[valid_mask]
        X_parts.append(win[valid_mask])
        meta_chunk = g.iloc[end_indices].copy()
        synth_arr = g[synth_label_col].to_numpy()
        type_arr = g[synth_type_col].to_numpy()
        window_labels, window_types, window_counts = [], [], []
        for end_idx in end_indices:
            start_idx = end_idx - window_size + 1
            window_synth = synth_arr[start_idx:end_idx + 1]
            window_type = type_arr[start_idx:end_idx + 1]
            has_synth = int(window_synth.max())
            n_synth = int(window_synth.sum())
            if has_synth:
                synth_positions = np.where(window_synth == 1)[0]
                wtype = window_type[synth_positions[-1]]
            else:
                wtype = "none"
            window_labels.append(has_synth)
            window_types.append(wtype)
            window_counts.append(n_synth)
        meta_chunk["window_synth_label"] = window_labels
        meta_chunk["window_synth_type"] = window_types
        meta_chunk["n_synth_in_window"] = window_counts
        meta_parts.append(meta_chunk)
    if X_parts:
        X = np.concatenate(X_parts, axis=0)
        meta = pd.concat(meta_parts, axis=0).reset_index(drop=True)
    else:
        X = np.empty((0, window_size, len(feature_cols)), dtype=np.float32)
        meta = pd.DataFrame()
    return X, meta


In [9]:
# 6 — Model Builder (Forecaster) + Training
def build_lstm_forecaster(n_input_features, n_output_features,
                          context_steps=23, latent_dim=32, n_layers=2, dropout=0.1):
    inputs = keras.Input(shape=(context_steps, n_input_features), name="forecast_input")
    x = inputs
    for i in range(n_layers):
        return_sequences = (i < n_layers - 1)
        x = layers.LSTM(latent_dim, return_sequences=return_sequences,
                        dropout=dropout, name=f"forecast_lstm_{i+1}")(x)
    outputs = layers.Dense(n_output_features, name="forecast_output")(x)
    return keras.Model(inputs, outputs, name="lstm_forecaster")

def train_model_generic(model, X_train, Y_train, X_val, Y_val,
                        epochs, lr, batch_size, early_stop, verbose=1):
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr, clipnorm=1.0), loss="mse")
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=early_stop,
                                      restore_best_weights=True, verbose=1),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4, verbose=1)
    ]
    history = model.fit(X_train, Y_train, validation_data=(X_val, Y_val),
                        epochs=epochs, batch_size=batch_size, callbacks=callbacks, verbose=verbose)
    return model, history


In [10]:
# 7 — Scoring Helpers (Forecaster)
def predict_in_batches(model, X, batch_size=2048):
    preds = []
    for i in range(0, len(X), batch_size):
        preds.append(model.predict(X[i:i + batch_size], verbose=0))
    return np.concatenate(preds, axis=0) if preds else np.empty((0, 0), dtype=np.float32)

def compute_forecast_scores(model, X_input, X_actual_last_step, score_features,
                            ae_features, mode="directional"):
    X_pred = predict_in_batches(model, X_input, batch_size=2048)
    sc_idx = [ae_features.index(f) for f in score_features]
    X_actual = X_actual_last_step[:, sc_idx]
    err = (X_pred - X_actual) ** 2
    if mode == "mean":
        return err.mean(axis=1)
    elif mode == "max":
        return err.max(axis=1)
    elif mode == "directional":
        abs_err = np.abs(X_pred - X_actual)
        denom = np.maximum(np.maximum(np.abs(X_actual), np.abs(X_pred)), 1e-6)
        rel_err = abs_err / denom
        return 0.5 * abs_err.mean(axis=1) + 0.5 * rel_err.mean(axis=1)
    else:
        raise ValueError(f"Unbekannter mode: {mode}")


In [11]:
# 8 — Z-Normalisierung
def znorm_score_by_hour(meta_df, raw_score_col, val_start, test_start,
                        new_col="score_znorm_hour"):
    """Hour-only Z-Norm. Returns (meta, mu_per_hour, std_per_hour)."""
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    mu_per_hour, std_per_hour = {}, {}
    for h in range(24):
        vals = meta.loc[val_normal & (hour_col == h), raw_score_col].dropna()
        mu_per_hour[h] = vals.mean() if len(vals) > 0 else 0.0
        std_per_hour[h] = vals.std() if len(vals) > 1 else 1.0
    hours = hour_col.values
    raw = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        znorm[i] = (raw[i] - mu_per_hour[hours[i]]) / max(std_per_hour[hours[i]], 1e-8)
    meta[new_col] = znorm
    return meta, mu_per_hour, std_per_hour

def znorm_score_by_hour_station(meta_df, raw_score_col, val_start, test_start,
                                new_col="score_znorm_hour_station"):
    """Hour x Station Z-Norm (with global fallback)."""
    meta = meta_df.copy()
    val_normal = (
        (meta["hour_ts"] >= val_start) &
        (meta["hour_ts"] < test_start) &
        (meta["synth_label"] == 0)
    )
    hour_col = meta["hour_ts"].dt.hour
    meta["_hour_tmp"] = hour_col
    stats = (meta[val_normal].groupby(["station_id", "_hour_tmp"])[raw_score_col]
             .agg(["mean", "std", "count"]).reset_index())
    global_stats = (meta[val_normal].groupby("_hour_tmp")[raw_score_col]
                    .agg(["mean", "std"]).reset_index()
                    .rename(columns={"mean": "g_mean", "std": "g_std"}))
    stats = stats.merge(global_stats, on="_hour_tmp", how="left")
    stats["use_mean"] = np.where(stats["count"] >= 10, stats["mean"], stats["g_mean"])
    stats["use_std"] = np.where(stats["count"] >= 10, stats["std"], stats["g_std"])
    stats["use_std"] = stats["use_std"].clip(lower=1e-8)
    meta = meta.merge(stats[["station_id", "_hour_tmp", "use_mean", "use_std"]],
                      on=["station_id", "_hour_tmp"], how="left")
    meta["use_mean"] = meta["use_mean"].fillna(0.0)
    meta["use_std"] = meta["use_std"].fillna(1.0).clip(lower=1e-8)
    meta[new_col] = (meta[raw_score_col] - meta["use_mean"]) / meta["use_std"]
    meta = meta.drop(columns=["_hour_tmp", "use_mean", "use_std"])
    return meta

def apply_znorm_with_external_stats(meta_df, raw_score_col, mu_per_hour, std_per_hour,
                                     new_col="score_znorm_transfer"):
    """Apply Z-Norm using pre-computed hour-level stats (from source city)."""
    meta = meta_df.copy()
    hours = meta["hour_ts"].dt.hour.values
    raw = meta[raw_score_col].values
    znorm = np.zeros_like(raw)
    for i in range(len(raw)):
        h = hours[i]
        znorm[i] = (raw[i] - mu_per_hour.get(h, 0.0)) / max(std_per_hour.get(h, 1.0), 1e-8)
    meta[new_col] = znorm
    return meta


In [12]:
# 9 — Synthetic Injection (ohne Contextuals)
V17B_INJECTION_PROBS = {"point_spike": 0.467, "point_drop": 0.400, "collective": 0.133}
V17B_COLLECTIVE_BLOCK_LEN = (2, 4)

def inject_synthetic_anomalies_v17b(df, inject_start, injection_rate=0.015,
                                     seed=42, verbose=True):
    probs = list(V17B_INJECTION_PROBS.values())
    type_names = list(V17B_INJECTION_PROBS.keys())
    block_min, block_max = V17B_COLLECTIVE_BLOCK_LEN
    rng = np.random.RandomState(seed)
    out = df.copy()
    out["original_n_lends"] = out["n_lends"].copy()
    out["original_n_returns"] = out["n_returns"].copy()
    out["synth_label"] = 0
    out["synth_type"] = "none"
    inject_mask = out["hour_ts"] >= inject_start
    inject_idx = out[inject_mask].index.to_numpy()
    if len(inject_idx) == 0:
        print("  WARNUNG: Keine Daten ab inject_start!")
        return out
    n_inject = max(1, int(len(inject_idx) * injection_rate))
    inject_df = out.loc[inject_idx]
    has_demand = inject_df[inject_df["total_demand"] > 0].index.to_numpy()
    if len(has_demand) < n_inject:
        n_inject = len(has_demand)
    chosen_idx = rng.choice(has_demand, size=n_inject, replace=False)
    types = rng.choice(type_names, size=n_inject, p=probs)
    injected_indices = set()
    requested_counts = {t: 0 for t in type_names}
    injected_counts = {t: 0 for t in type_names}
    for idx, anom_type in zip(chosen_idx, types):
        requested_counts[anom_type] += 1
        if idx in injected_indices:
            continue
        row = out.loc[idx]
        if anom_type == "point_spike":
            factor = rng.uniform(3.0, 8.0)
            out.loc[idx, "n_lends"] = int(row["n_lends"] * factor) + rng.randint(2, 10)
            out.loc[idx, "n_returns"] = int(row["n_returns"] * factor) + rng.randint(2, 10)
            out.loc[idx, "synth_label"] = 1
            out.loc[idx, "synth_type"] = "point_spike"
            injected_indices.add(idx)
            injected_counts["point_spike"] += 1
        elif anom_type == "point_drop":
            regime = row.get("demand_regime", "low")
            do_drop = (regime == "high" and row["total_demand"] >= 8) or                       (regime == "mid" and row["total_demand"] >= 5)
            if do_drop:
                out.loc[idx, "n_lends"] = rng.randint(0, 2)
                out.loc[idx, "n_returns"] = rng.randint(0, 2)
                out.loc[idx, "synth_label"] = 1
                out.loc[idx, "synth_type"] = "point_drop"
                injected_indices.add(idx)
                injected_counts["point_drop"] += 1
        elif anom_type == "collective":
            block_len = rng.randint(block_min, block_max + 1)
            sid = row["station_id"]
            station_inject = out[(out["station_id"] == sid) & inject_mask].sort_values("hour_ts")
            if idx in station_inject.index:
                pos = station_inject.index.get_loc(idx)
                if pos + block_len <= len(station_inject):
                    block_idx = station_inject.index[pos:pos + block_len]
                    factor = rng.uniform(2.5, 5.0)
                    block_actually = 0
                    for bidx in block_idx:
                        if bidx not in injected_indices:
                            out.loc[bidx, "n_lends"] = int(out.loc[bidx, "original_n_lends"] * factor) + rng.randint(1, 5)
                            out.loc[bidx, "n_returns"] = int(out.loc[bidx, "original_n_returns"] * factor) + rng.randint(1, 5)
                            out.loc[bidx, "synth_label"] = 1
                            out.loc[bidx, "synth_type"] = "collective"
                            injected_indices.add(bidx)
                            block_actually += 1
                    injected_counts["collective"] += block_actually
    out["total_demand"] = out["n_lends"] + out["n_returns"]
    out["net_flow"] = out["n_returns"] - out["n_lends"]
    out["abs_net_flow"] = out["net_flow"].abs()
    n_injected = out["synth_label"].sum()
    inject_total = inject_mask.sum()
    if verbose:
        print(f"\n  Synthetic Injection (ohne Contextuals):")
        print(f"    Inject-Zeilen gesamt:  {inject_total:,}")
        print(f"    Injizierte Punkte:     {n_injected:,} ({n_injected/inject_total*100:.2f}%)")
        print(f"\n    Requested vs Actually Injected:")
        for t in type_names:
            print(f"      {t:15s}  requested={requested_counts[t]:5d}  injected={injected_counts[t]:5d}")
        print(f"\n    Effective rates (of inject region):")
        for t in type_names:
            rate = injected_counts[t] / inject_total * 100
            print(f"      {t:15s}  {rate:.3f}%")
    return out


In [13]:
# 10 — City Data Pipeline
def prepare_city_data(demand_path, geo_path, station_names_path, cfg, city_name):
    print(f"\n{'='*70}")
    print(f"  DATEN-PIPELINE: {city_name}")
    print(f"{'='*70}")
    demand = pd.read_parquet(demand_path)
    geo = pd.read_parquet(geo_path)
    snames = pd.read_parquet(station_names_path)
    snames = snames.rename(columns={'id': 'station_name_id', 'name': 'station_name'})
    demand = demand.merge(snames[['station_name_id', 'station_name']], on='station_name_id', how='left')
    demand = demand.merge(geo[['location_id', 'latitude', 'longitude']], on='location_id', how='left')
    demand['timestamp'] = pd.to_datetime(demand['timestamp'], utc=True)
    print(f"  Roh: {len(demand):,} Zeilen, {demand['station_id'].nunique()} Stationen")
    print(f"  Zeitraum: {demand['timestamp'].min().date()} bis {demand['timestamp'].max().date()}")
    print("\n  [1] Aggregation...")
    df = aggregate_station_level(demand, minutes=cfg.aggregation_minutes)
    print(f"      Shape: {df.shape}")
    print("  [2] Gap-Fill...")
    n_before = len(df)
    df = fill_missing_time_bins(df, minutes=cfg.aggregation_minutes)
    print(f"      {n_before:,} -> {len(df):,} (+{len(df)-n_before:,})")
    print("  [3] Stationen, Regime, Splits...")
    df, station_profile, train_end, val_end = prepare_stations_and_splits(df, cfg, city_name)
    print("  [4] Statistische Scoring-Pipeline...")
    df = run_statistical_pipeline(df, cfg, train_end)
    return df, station_profile, train_end, val_end


In [14]:
# 11 — Unified Evaluation (FC only, mit Regime x Type Matrix)
ANOM_TYPES = ["point_spike", "point_drop", "collective"]
REGIMES = ["high", "mid", "low"]

def evaluate_v17c(meta, score_col, experiment_name, val_start, test_start,
                  fixed_threshold=None, verbose=True):
    """Evaluate with optional fixed_threshold (for Zero-Shot with source threshold)."""
    meta = meta.copy()
    meta["split_eval"] = np.where(
        meta["hour_ts"] < val_start, "train",
        np.where(meta["hour_ts"] < test_start, "val", "test")
    )
    val_m = meta[meta["split_eval"] == "val"].copy()
    test_m = meta[meta["split_eval"] == "test"].copy()
    if len(val_m) == 0 or len(test_m) == 0:
        print(f"  [{experiment_name}] WARNUNG: Keine VAL/TEST-Daten!")
        return {}
    results = {"experiment": experiment_name, "n_val": len(val_m), "n_test": len(test_m)}
    # Anomaly Rates
    n_test_total = len(test_m)
    n_test_anom = (test_m["synth_label"] == 1).sum()
    results["anomaly_rate_total"] = n_test_anom / n_test_total
    for atype in ANOM_TYPES:
        n_t = (test_m["synth_type"] == atype).sum()
        results[f"rate_{atype}"] = n_t / n_test_total
        results[f"n_{atype}"] = int(n_t)
    # Threshold
    if fixed_threshold is not None:
        threshold = fixed_threshold
        results["threshold_source"] = "source_val"
    else:
        y_val = val_m["synth_label"].astype(int).values
        s_val = val_m[score_col].values
        threshold = None
        if len(np.unique(y_val)) > 1:
            prec_v, rec_v, thr_v = precision_recall_curve(y_val, s_val)
            results["val_pr_auc"] = average_precision_score(y_val, s_val)
            if len(thr_v) > 0:
                f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
                best_idx = np.argmax(f1_v)
                threshold = float(thr_v[best_idx])
                results["val_best_f1"] = float(f1_v[best_idx])
        if threshold is None:
            val_norm = val_m[val_m["synth_label"] == 0][score_col]
            threshold = float(val_norm.quantile(0.99)) if len(val_norm) > 0 else float(test_m[score_col].quantile(0.99))
            results["threshold_source"] = "fallback_99pct"
    results["val_threshold"] = threshold
    # TEST Global
    y_test = test_m["synth_label"].astype(int).values
    s_test = test_m[score_col].values
    p_test = (s_test >= threshold).astype(int)
    if len(np.unique(y_test)) > 1:
        results["test_pr_auc"] = average_precision_score(y_test, s_test)
        results["test_roc_auc"] = roc_auc_score(y_test, s_test)
        results["test_f1"] = f1_score(y_test, p_test, zero_division=0)
        results["test_prec"] = precision_score(y_test, p_test, zero_division=0)
        results["test_recall"] = recall_score(y_test, p_test, zero_division=0)
    # Per-Type
    for atype in ANOM_TYPES:
        type_mask = test_m["synth_type"] == atype
        normal_mask = test_m["synth_label"] == 0
        sub = test_m[type_mask | normal_mask].copy()
        n_anom = int(type_mask.sum())
        if n_anom > 0 and len(sub) > 0:
            y_t = (sub["synth_type"] == atype).astype(int).values
            y_s = sub[score_col].values
            if len(set(y_t)) > 1:
                results[f"pr_{atype}"] = average_precision_score(y_t, y_s)
            type_seqs = test_m[type_mask]
            detected = (type_seqs[score_col] >= threshold).sum()
            results[f"dr_{atype}"] = detected / len(type_seqs) if len(type_seqs) > 0 else 0.0
        else:
            results[f"pr_{atype}"] = None
            results[f"dr_{atype}"] = None
    # Per-Type x Per-Regime (3x3 Matrix)
    regime_rows = []
    for atype in ANOM_TYPES:
        for regime in REGIMES:
            rm = test_m["demand_regime"] == regime
            tm = test_m["synth_type"] == atype
            nm = test_m["synth_label"] == 0
            n_a = int((rm & tm).sum())
            n_n = int((rm & nm).sum())
            pr, f1_val, dr = None, None, None
            if n_a > 0 and n_n > 0:
                sub = test_m[(rm & tm) | (rm & nm)]
                y_t = (sub["synth_type"] == atype).astype(int).values
                y_s = sub[score_col].values
                pr = average_precision_score(y_t, y_s) if len(set(y_t)) > 1 else None
                p_sub = (y_s >= threshold).astype(int)
                f1_val = f1_score(y_t, p_sub, zero_division=0)
                detected = (test_m[rm & tm][score_col] >= threshold).sum()
                dr = detected / n_a
            regime_rows.append({"type": atype, "regime": regime, "n_anom": n_a, "pr_auc": pr, "f1": f1_val, "dr": dr})
    results["regime_detail"] = pd.DataFrame(regime_rows)
    if verbose:
        print(f"\n  [{experiment_name}]")
        print(f"    Anomaly Rate: {results['anomaly_rate_total']:.4f} ({results['anomaly_rate_total']*100:.2f}%)")
        for at in ANOM_TYPES:
            print(f"      {at:15s}: n={results[f'n_{at}']:5d}, rate={results[f'rate_{at}']:.4f}")
        print(f"    Threshold: {threshold:.6f} (source: {results.get('threshold_source', 'val_best_f1')})")
        pr_auc_v = results.get('test_pr_auc')
        f1_v = results.get('test_f1')
        prec_v = results.get('test_prec')
        rec_v = results.get('test_recall')
        print(f"    TEST PR-AUC={pr_auc_v:.4f}, F1={f1_v:.4f}, Prec={prec_v:.4f}, Recall={rec_v:.4f}")
        print(f"    Per-Type PR-AUC:")
        for at in ANOM_TYPES:
            v = results.get(f'pr_{at}')
            print(f"      {at:15s}: {v:.4f}" if v is not None else f"      {at:15s}: N/A")
        print(f"\n    Type x Regime Matrix (PR-AUC / F1):")
        rd = results["regime_detail"]
        for regime in REGIMES:
            vals = []
            for atype in ANOM_TYPES:
                row = rd[(rd["type"] == atype) & (rd["regime"] == regime)].iloc[0]
                pr_s = f"{row['pr_auc']:.3f}" if row['pr_auc'] is not None else "  N/A"
                f1_s = f"{row['f1']:.3f}" if row['f1'] is not None else "  N/A"
                vals.append(f"{pr_s}/{f1_s}")
            print(f"      {regime:4s}: " + "  |  ".join(vals))
        print(f"            {'spike':>11s}  |  {'drop':>11s}  |  {'collective':>11s}")
    return results


In [15]:
# 12 — Sequenzen + Scaler Helper
ae_features = cfg.ae_features
score_features = cfg.ae_score_features
context_steps = cfg.ae_window_size - 1  # 23
score_idx = [ae_features.index(f) for f in score_features]

def build_sequences_for_city(df_clean, df_inj, train_end, val_end, city_name, scaler_ext=None):
    df_c = df_clean.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_c = df_c.dropna(subset=ae_features)
    train_mask_sc = df_c["hour_ts"] < train_end
    if scaler_ext is None:
        scaler = StandardScaler()
        scaler.fit(df_c.loc[train_mask_sc, ae_features])
        print(f"  [{city_name}] Scaler gefittet auf {train_mask_sc.sum():,} Train-Zeilen")
    else:
        scaler = scaler_ext
        print(f"  [{city_name}] Externer Scaler verwendet (Transfer)")
    df_inj_s = df_inj.copy().sort_values(["station_id", "hour_ts"]).reset_index(drop=True)
    df_inj_s = df_inj_s.dropna(subset=ae_features)
    df_inj_scaled = df_inj_s.copy()
    df_inj_scaled[ae_features] = scaler.transform(df_inj_s[ae_features])
    for col in ["synth_label", "synth_type", "original_n_lends", "original_n_returns",
                "demand_regime", "label_eval"]:
        if col in df_inj_s.columns:
            df_inj_scaled[col] = df_inj_s[col].values
    X_all, meta_all = make_sequences_with_window_labels(
        df_inj_scaled, feature_cols=ae_features, window_size=cfg.ae_window_size,
        synth_label_col="synth_label", synth_type_col="synth_type",
        require_contiguous=cfg.require_contiguous, agg_minutes=cfg.aggregation_minutes
    )
    meta_all["split_eval"] = np.where(
        meta_all["hour_ts"] < train_end, "train",
        np.where(meta_all["hour_ts"] < val_end, "val", "test")
    )
    print(f"  [{city_name}] Sequenzen: {X_all.shape}")
    print(f"  [{city_name}] Split: {meta_all['split_eval'].value_counts().to_dict()}")
    train_normal = (meta_all["split_eval"] == "train") & (meta_all["label_eval"] == "normal")
    val_normal = (meta_all["split_eval"] == "val") & (meta_all["synth_label"] == 0) & (meta_all["label_eval"] == "normal")
    X_train = X_all[train_normal.values]
    X_val_c = X_all[val_normal.values]
    print(f"  [{city_name}] X_train (normal): {X_train.shape}, X_val (normal): {X_val_c.shape}")
    return {
        "X_all": X_all, "meta_all": meta_all, "scaler": scaler,
        "X_train": X_train, "X_val_clean": X_val_c,
        "train_normal_mask": train_normal, "val_normal_mask": val_normal,
        "train_end": train_end, "val_end": val_end,
    }


---
## Schritt 1: Alle Staedte laden + Injection


In [16]:
# 13 — Alle benoetigten Staedte laden + injizieren
all_cities_needed = set()
for target, pairs in EXPERIMENT_PAIRS.items():
    all_cities_needed.add(target)
    for source, _ in pairs:
        all_cities_needed.add(source)
print(f"Benoetigte Staedte: {sorted(all_cities_needed)}\n")

city_raw = {}
city_injected = {}

for i, city in enumerate(sorted(all_cities_needed)):
    if city not in CITY_DEMAND:
        print(f"\n  *** FEHLT: {city} nicht in CITY_DEMAND! ***")
        continue
    df, profile, train_end, val_end = prepare_city_data(
        CITY_DEMAND[city], geo_path, station_names_path, cfg, city
    )
    city_raw[city] = (df, profile, train_end, val_end)
    seed = cfg.injection_seed + i
    df_inj = inject_synthetic_anomalies_v17b(
        df, inject_start=train_end, injection_rate=cfg.injection_rate,
        seed=seed, verbose=True
    )
    city_injected[city] = df_inj
    vm = (df_inj["hour_ts"] >= train_end) & (df_inj["hour_ts"] < val_end)
    tm = df_inj["hour_ts"] >= val_end
    print(f"\n  {city}: VAL-Anomalien={df_inj.loc[vm, 'synth_label'].sum():,}, "
          f"TEST-Anomalien={df_inj.loc[tm, 'synth_label'].sum():,}")

print(f"\n{'='*70}")
print(f"  Alle {len(city_raw)} Staedte geladen und injiziert.")
print(f"{'='*70}")


Benoetigte Staedte: ['Braunschweig', 'Freiburg', 'Heidelberg', 'Kaiserslautern', 'Ludwigshafen', 'Mannheim', 'Marburg']


  DATEN-PIPELINE: Braunschweig
  Roh: 397,269 Zeilen, 159 Stationen
  Zeitraum: 2025-02-03 bis 2026-02-02

  [1] Aggregation...
      Shape: (227493, 22)
  [2] Gap-Fill...
      227,493 -> 984,904 (+757,411)
  [3] Stationen, Regime, Splits...
  Aktive Stationen (Braunschweig): 88
  Regime: {'low': 32, 'high': 32, 'mid': 31}
  Zeitraum: 2025-02-03 bis 2026-02-02 (365 Tage)
  Split: Train < 2025-10-05, Val < 2025-11-29, Test ab 2025-11-29
  [4] Statistische Scoring-Pipeline...
  [1/5] Kontext-Keys...
  [2/5] Rolling z-Scores...
  [3/5] Poisson-Tail-Scores...
  [4/5] ECDF-Kalibrierung...
  [5/5] Labels...
  Label-Verteilung:
    {'normal': 678247, 'grauzone_low': 18032, 'grauzone_high': 14160, 'anomal_high': 1126, 'anomal_low': 181}
    Anomalie-Rate: 0.0018 (0.18%)

  Synthetic Injection (ohne Contextuals):
    Inject-Zeilen gesamt:  273,652
    Injizierte Punkte:    

---
## Schritt 2: Sequenzen fuer alle Staedte


In [17]:
# 14 — Sequenzen fuer alle Staedte bauen
city_data = {}
for city in sorted(city_raw.keys()):
    df, profile, train_end, val_end = city_raw[city]
    df_inj = city_injected[city]
    print(f"\n--- {city} ---")
    city_data[city] = build_sequences_for_city(df, df_inj, train_end, val_end, city)
print(f"\nSequenzen fuer {len(city_data)} Staedte gebaut.")



--- Braunschweig ---
  [Braunschweig] Scaler gefittet auf 438,094 Train-Zeilen
  [Braunschweig] Sequenzen: (595091, 24, 9)
  [Braunschweig] Split: {'train': 361877, 'test': 126861, 'val': 106353}
  [Braunschweig] X_train (normal): (347841, 24, 9), X_val (normal): (98995, 24, 9)

--- Freiburg ---
  [Freiburg] Scaler gefittet auf 572,802 Train-Zeilen
  [Freiburg] Sequenzen: (662196, 24, 9)
  [Freiburg] Split: {'train': 443784, 'test': 118377, 'val': 100035}
  [Freiburg] X_train (normal): (428588, 24, 9), X_val (normal): (94869, 24, 9)

--- Heidelberg ---
  [Heidelberg] Scaler gefittet auf 1,115,042 Train-Zeilen
  [Heidelberg] Sequenzen: (1063300, 24, 9)
  [Heidelberg] Split: {'train': 696373, 'test': 197782, 'val': 169145}
  [Heidelberg] X_train (normal): (671998, 24, 9), X_val (normal): (160296, 24, 9)

--- Kaiserslautern ---
  [Kaiserslautern] Scaler gefittet auf 400,817 Train-Zeilen
  [Kaiserslautern] Sequenzen: (547922, 24, 9)
  [Kaiserslautern] Split: {'train': 366901, 'test': 9874

---
## Experiment-Runner-Funktionen


In [18]:
# 15 — Experiment Runner: Oracle, Target-Only, Zero-Shot, Fine-Tune

def run_fc_train_and_score(cdata, cfg, city_name, model=None, train_fraction=1.0, subsample_seed=42):
    """Train (or reuse) a FC model and score all sequences."""
    X_train = cdata["X_train"]
    X_val_c = cdata["X_val_clean"]
    X_all = cdata["X_all"]
    meta = cdata["meta_all"].copy()
    train_end = cdata["train_end"]
    val_end = cdata["val_end"]
    if train_fraction < 1.0:
        rng = np.random.RandomState(subsample_seed)
        n_total = len(X_train)
        n_sub = max(1, int(n_total * train_fraction))
        sub_idx = rng.choice(n_total, size=n_sub, replace=False)
        sub_idx.sort()
        X_train_use = X_train[sub_idx]
        print(f"  [{city_name}] Subsample: {n_sub}/{n_total} ({train_fraction*100:.0f}%)")
    else:
        X_train_use = X_train
    if model is None:
        print(f"  [{city_name}] Training FC...")
        model = build_lstm_forecaster(
            n_input_features=len(ae_features), n_output_features=len(score_features),
            context_steps=context_steps, latent_dim=cfg.ae_latent_dim,
            n_layers=2, dropout=cfg.ae_dropout,
        )
        X_fc_train = X_train_use[:, :context_steps, :]
        Y_fc_train = X_train_use[:, -1, :][:, score_idx]
        X_fc_val = X_val_c[:, :context_steps, :]
        Y_fc_val = X_val_c[:, -1, :][:, score_idx]
        model, _ = train_model_generic(
            model, X_fc_train, Y_fc_train, X_fc_val, Y_fc_val,
            epochs=cfg.ae_epochs, lr=cfg.ae_lr,
            batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop, verbose=0
        )
        print(f"  [{city_name}] FC trainiert.")
    else:
        print(f"  [{city_name}] Vorhandenes FC-Modell verwendet.")
    X_fc_input = X_all[:, :context_steps, :]
    X_actual_last = X_all[:, -1, :]
    meta["score_fc_raw"] = compute_forecast_scores(
        model, X_fc_input, X_actual_last, score_features, ae_features, mode="directional"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_fc_raw", train_end, val_end, new_col="score_fc_znorm_hs"
    )
    return model, meta


def run_oracle(target_city, cfg):
    """Oracle 100%: Train FC on full target train data."""
    print(f"\n{'='*60}")
    print(f"  ORACLE 100%: {target_city}")
    print(f"{'='*60}")
    cdata = city_data[target_city]
    model, meta = run_fc_train_and_score(cdata, cfg, target_city)
    res = evaluate_v17c(
        meta, "score_fc_znorm_hs", f"{target_city}_Oracle",
        val_start=cdata["train_end"], test_start=cdata["val_end"]
    )
    model.save(f"{MODELS_DIR}/{target_city.lower()}_oracle_fc.keras")
    return res, model, meta


def run_target_only_5pct(target_city, cfg):
    """Target-only 5%: Train FC on 5% of target train data."""
    print(f"\n{'='*60}")
    print(f"  TARGET-ONLY 5%: {target_city}")
    print(f"{'='*60}")
    cdata = city_data[target_city]
    model, meta = run_fc_train_and_score(cdata, cfg, target_city, train_fraction=cfg.finetune_ratio)
    res = evaluate_v17c(
        meta, "score_fc_znorm_hs", f"{target_city}_TargetOnly5",
        val_start=cdata["train_end"], test_start=cdata["val_end"]
    )
    return res


def run_zero_shot_strict(source_city, target_city, pair_type, cfg, source_model, source_info):
    """Strict Pure Zero-Shot: Source model + source scaler + source Z-Norm + source threshold."""
    print(f"\n{'='*60}")
    print(f"  ZERO-SHOT STRICT: {source_city} ({pair_type}) -> {target_city}")
    print(f"{'='*60}")
    df_target, _, train_end_t, val_end_t = city_raw[target_city]
    df_target_inj = city_injected[target_city]
    print(f"  Target-Daten mit {source_city}-Scaler skalieren...")
    target_zs_data = build_sequences_for_city(
        df_target, df_target_inj, train_end_t, val_end_t,
        f"{target_city}_ZS", scaler_ext=source_info["scaler"]
    )
    X_all_zs = target_zs_data["X_all"]
    meta_zs = target_zs_data["meta_all"].copy()
    X_fc_input = X_all_zs[:, :context_steps, :]
    X_actual = X_all_zs[:, -1, :]
    meta_zs["score_fc_raw"] = compute_forecast_scores(
        source_model, X_fc_input, X_actual, score_features, ae_features, mode="directional"
    )
    # Z-Norm with SOURCE hour-level stats
    src_meta = source_info["meta_all_scored"]
    src_val_normal = (
        (src_meta["hour_ts"] >= source_info["train_end"]) &
        (src_meta["hour_ts"] < source_info["val_end"]) &
        (src_meta["synth_label"] == 0)
    )
    src_hour_col = src_meta["hour_ts"].dt.hour
    src_mu, src_std = {}, {}
    for h in range(24):
        vals = src_meta.loc[src_val_normal & (src_hour_col == h), "score_fc_raw"].dropna()
        src_mu[h] = vals.mean() if len(vals) > 0 else 0.0
        src_std[h] = vals.std() if len(vals) > 1 else 1.0
    meta_zs = apply_znorm_with_external_stats(
        meta_zs, "score_fc_raw", src_mu, src_std, new_col="score_fc_znorm_zs"
    )
    # Threshold from source val
    src_meta_znorm = apply_znorm_with_external_stats(
        src_meta, "score_fc_raw", src_mu, src_std, new_col="score_fc_znorm_src"
    )
    src_val_mask = (src_meta["hour_ts"] >= source_info["train_end"]) & (src_meta["hour_ts"] < source_info["val_end"])
    src_val_scores = src_meta_znorm.loc[src_val_mask, "score_fc_znorm_src"].values
    src_val_labels = src_meta.loc[src_val_mask, "synth_label"].astype(int).values
    if len(np.unique(src_val_labels)) > 1:
        prec_v, rec_v, thr_v = precision_recall_curve(src_val_labels, src_val_scores)
        f1_v = 2 * prec_v[:-1] * rec_v[:-1] / (prec_v[:-1] + rec_v[:-1] + 1e-12)
        best_idx = np.argmax(f1_v)
        src_threshold = float(thr_v[best_idx])
    else:
        src_norm_mask = src_val_mask & (src_meta["synth_label"] == 0)
        src_threshold = float(src_meta_znorm.loc[src_norm_mask, "score_fc_znorm_src"].quantile(0.99))
    print(f"  Source-Threshold ({source_city}): {src_threshold:.6f}")
    res = evaluate_v17c(
        meta_zs, "score_fc_znorm_zs",
        f"ZS_{source_city}({pair_type})->{target_city}",
        val_start=train_end_t, test_start=val_end_t, fixed_threshold=src_threshold
    )
    return res


def run_finetune_5pct(source_city, target_city, pair_type, cfg, source_model, source_info):
    """Fine-Tune 5%: Pre-trained source model, fine-tuned with 5% target train."""
    print(f"\n{'='*60}")
    print(f"  FINE-TUNE 5%: {source_city} ({pair_type}) -> {target_city}")
    print(f"{'='*60}")
    cdata = city_data[target_city]
    X_train = cdata["X_train"]
    X_val_c = cdata["X_val_clean"]
    X_all = cdata["X_all"]
    meta = cdata["meta_all"].copy()
    train_end = cdata["train_end"]
    val_end = cdata["val_end"]
    rng_ft = np.random.RandomState(42)
    n_total = len(X_train)
    n_ft = max(1, int(n_total * cfg.finetune_ratio))
    ft_idx = rng_ft.choice(n_total, size=n_ft, replace=False)
    ft_idx.sort()
    X_ft_train = X_train[ft_idx]
    X_ft_val = X_val_c
    print(f"  Fine-Tune Training: {n_ft}/{n_total} ({cfg.finetune_ratio*100:.0f}%)")
    fc_ft = keras.models.clone_model(source_model)
    fc_ft.set_weights(source_model.get_weights())
    X_fc_ft_train = X_ft_train[:, :context_steps, :]
    Y_fc_ft_train = X_ft_train[:, -1, :][:, score_idx]
    X_fc_ft_val = X_ft_val[:, :context_steps, :]
    Y_fc_ft_val = X_ft_val[:, -1, :][:, score_idx]
    fc_ft, _ = train_model_generic(
        fc_ft, X_fc_ft_train, Y_fc_ft_train, X_fc_ft_val, Y_fc_ft_val,
        epochs=cfg.ae_epochs, lr=cfg.ae_lr * 0.1,
        batch_size=cfg.ae_batch_size, early_stop=cfg.ae_early_stop, verbose=0
    )
    X_fc_input = X_all[:, :context_steps, :]
    X_actual_last = X_all[:, -1, :]
    meta["score_fc_raw"] = compute_forecast_scores(
        fc_ft, X_fc_input, X_actual_last, score_features, ae_features, mode="directional"
    )
    meta = znorm_score_by_hour_station(
        meta, "score_fc_raw", train_end, val_end, new_col="score_fc_znorm_ft"
    )
    res = evaluate_v17c(
        meta, "score_fc_znorm_ft",
        f"FT5_{source_city}({pair_type})->{target_city}",
        val_start=train_end, test_start=val_end
    )
    return res


---
## Schritt 3: Source-Modelle trainieren / laden


In [19]:
# 16 — Source-Modelle: Trainieren oder Laden
CITY_ABBR = {
    "Mannheim": "ma",
    "Heidelberg": "hd",
    "Freiburg": "fr",
    "Braunschweig": "bs",
    "Marburg": "mr",
    "Ludwigshafen": "lu",
    "Kaiserslautern": "kl",
}

source_cities_needed = set()
for target, pairs in EXPERIMENT_PAIRS.items():
    for source, _ in pairs:
        source_cities_needed.add(source)
print(f"Benoetigte Source-Modelle: {sorted(source_cities_needed)}")

source_models = {}

for src_city in sorted(source_cities_needed):
    print(f"\n{'='*70}")
    print(f"  SOURCE-MODELL: {src_city}")
    print(f"{'='*70}")
    cdata = city_data[src_city]
    loaded = False

    abbr = CITY_ABBR.get(src_city, src_city.lower())

    possible_paths = [
        f"{V17B_MODELS}/{abbr}_oracle_forecaster.keras",
        f"{MODELS_DIR}/{src_city.lower()}_source_fc.keras",
    ]

    # Optional: altes generisches V17-Modell nur fuer Mannheim als Fallback
    if src_city == "Mannheim":
        possible_paths.append(f"{V17_MODELS}/lstm_forecaster.keras")

    fc_model = None
    for model_path in possible_paths:
        if os.path.exists(model_path):
            try:
                fc_model = keras.models.load_model(model_path, compile=False)
                print(f"  Modell GELADEN: {model_path}")
                loaded = True
                break
            except Exception as e:
                print(f"  Laden fehlgeschlagen ({model_path}): {e}")

    if fc_model is None:
        print(f"  -> Kein gespeichertes Modell gefunden. Training...")
        fc_model, meta_scored = run_fc_train_and_score(cdata, cfg, src_city)
        fc_model.save(f"{MODELS_DIR}/{src_city.lower()}_source_fc.keras")
        print(f"  Modell trainiert und gespeichert.")
    else:
        _, meta_scored = run_fc_train_and_score(cdata, cfg, src_city, model=fc_model)

    source_models[src_city] = {
        "model": fc_model,
        "scaler": cdata["scaler"],
        "meta_all_scored": meta_scored,
        "train_end": cdata["train_end"],
        "val_end": cdata["val_end"],
        "loaded": loaded,
    }

print(f"\n{'='*70}")
print(f"  {len(source_models)} Source-Modelle bereit.")
for c, info in source_models.items():
    status = "GELADEN" if info["loaded"] else "NEU TRAINIERT"
    print(f"    {c:18s}: {status}")
print(f"{'='*70}")


Benoetigte Source-Modelle: ['Heidelberg', 'Kaiserslautern', 'Ludwigshafen', 'Mannheim', 'Marburg']

  SOURCE-MODELL: Heidelberg
  Modell GELADEN: /content/drive/MyDrive/BA_Colab/data/v17b_transfer_sanity/models/hd_oracle_forecaster.keras
  [Heidelberg] Vorhandenes FC-Modell verwendet.

  SOURCE-MODELL: Kaiserslautern
  -> Kein gespeichertes Modell gefunden. Training...
  [Kaiserslautern] Training FC...

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 37: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 41: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Epoch 41: early stopping
Restoring model weights from the end of the best epoch: 33.
  [Kaiserslautern] FC trainiert.
  Modell trainiert und gespeichert.

  SOURCE-MODELL: Ludwigshafen
  -> Kein gespeichertes Modell gefunden. Training...
  [Ludwigshafen] Training FC...

Epoch 20: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 25: R

---
## Schritt 4: Alle Experimente durchfuehren (3 Targets x 6 Ergebnisse)


In [20]:
# 17 — Alle 18 Experimente durchfuehren
all_results = []

for target_city, pairs in EXPERIMENT_PAIRS.items():
    print(f"\n{'#'*70}")
    print(f"  TARGET: {target_city}")
    print(f"{'#'*70}")

    # 1) Oracle 100%
    oracle_res, oracle_model, oracle_meta = run_oracle(target_city, cfg)
    all_results.append({
        "Target": target_city, "Source": "-", "PairType": "baseline",
        "Mode": "Oracle100",
        **{k: v for k, v in oracle_res.items() if k != "regime_detail"},
        "regime_detail": oracle_res.get("regime_detail"),
    })

    # 2) Target-only 5%
    to5_res = run_target_only_5pct(target_city, cfg)
    all_results.append({
        "Target": target_city, "Source": "-", "PairType": "baseline",
        "Mode": "TargetOnly5",
        **{k: v for k, v in to5_res.items() if k != "regime_detail"},
        "regime_detail": to5_res.get("regime_detail"),
    })

    # 3-6) Zero-Shot + Fine-Tune per source pair
    for source_city, pair_type in pairs:
        src_info = source_models[source_city]

        zs_res = run_zero_shot_strict(
            source_city, target_city, pair_type, cfg,
            src_info["model"], src_info
        )
        all_results.append({
            "Target": target_city, "Source": source_city, "PairType": pair_type,
            "Mode": "ZeroShot",
            **{k: v for k, v in zs_res.items() if k != "regime_detail"},
            "regime_detail": zs_res.get("regime_detail"),
        })

        ft_res = run_finetune_5pct(
            source_city, target_city, pair_type, cfg,
            src_info["model"], src_info
        )
        all_results.append({
            "Target": target_city, "Source": source_city, "PairType": pair_type,
            "Mode": "FT5",
            **{k: v for k, v in ft_res.items() if k != "regime_detail"},
            "regime_detail": ft_res.get("regime_detail"),
        })

print(f"\n{'='*70}")
print(f"  ALLE {len(all_results)} EXPERIMENTE ABGESCHLOSSEN")
print(f"{'='*70}")



######################################################################
  TARGET: Mannheim
######################################################################

  ORACLE 100%: Mannheim
  [Mannheim] Training FC...

Epoch 21: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.

Epoch 36: ReduceLROnPlateau reducing learning rate to 0.0002500000118743628.

Epoch 46: ReduceLROnPlateau reducing learning rate to 0.0001250000059371814.
Restoring model weights from the end of the best epoch: 47.
  [Mannheim] FC trainiert.

  [Mannheim_Oracle]
    Anomaly Rate: 0.0127 (1.27%)
      point_spike    : n= 2373, rate=0.0064
      point_drop     : n=  200, rate=0.0005
      collective     : n= 2095, rate=0.0057
    Threshold: 4.712992 (source: val_best_f1)
    TEST PR-AUC=0.6614, F1=0.6584, Prec=0.7256, Recall=0.6026
    Per-Type PR-AUC:
      point_spike    : 0.8277
      point_drop     : 0.0031
      collective     : 0.2708

    Type x Regime Matrix (PR-AUC / F1):
      high: 0.890/

---
## Zusammenfassung / Summary


In [21]:
# 18 — V17c Gesamtvergleich
rows = []
for r in all_results:
    rows.append({
        "Target": r["Target"], "Source": r["Source"],
        "PairType": r["PairType"], "Mode": r["Mode"],
        "PR-AUC": r.get("test_pr_auc"), "F1": r.get("test_f1"),
        "Precision": r.get("test_prec"), "Recall": r.get("test_recall"),
        "Threshold": r.get("val_threshold"),
        "Thr_Source": r.get("threshold_source", "val_best_f1"),
        "PR_spike": r.get("pr_point_spike"), "PR_drop": r.get("pr_point_drop"),
        "PR_coll": r.get("pr_collective"),
        "DR_spike": r.get("dr_point_spike"), "DR_drop": r.get("dr_point_drop"),
        "DR_coll": r.get("dr_collective"),
        "AnomRate": r.get("anomaly_rate_total"),
        "Rate_spike": r.get("rate_point_spike"), "Rate_drop": r.get("rate_point_drop"),
        "Rate_coll": r.get("rate_collective"),
        "N_spike": r.get("n_point_spike"), "N_drop": r.get("n_point_drop"),
        "N_coll": r.get("n_collective"),
    })
df_summary = pd.DataFrame(rows)

print("=" * 130)
print("  V17c GESAMTVERGLEICH -- Cross-City Pair Comparison")
print("=" * 130)

print("\n--- Hauptmetriken ---")
main_cols = ["Target", "Source", "PairType", "Mode", "PR-AUC", "F1", "Precision", "Recall", "Threshold"]
print(df_summary[main_cols].to_string(index=False, float_format="%.4f"))

print("\n--- Anomaly Rates im Testset ---")
rate_cols = ["Target", "Mode", "AnomRate", "Rate_spike", "Rate_drop", "Rate_coll",
             "N_spike", "N_drop", "N_coll"]
df_rates = df_summary[rate_cols].drop_duplicates(subset=["Target", "Mode"]).copy()
for col in ["AnomRate", "Rate_spike", "Rate_drop", "Rate_coll"]:
    df_rates[col] = df_rates[col].apply(lambda x: f"{x*100:.3f}%" if pd.notna(x) else "N/A")
print(df_rates.to_string(index=False))

print("\n--- Per-Type PR-AUC ---")
type_cols = ["Target", "Source", "PairType", "Mode", "PR_spike", "PR_drop", "PR_coll"]
print(df_summary[type_cols].to_string(index=False, float_format="%.4f"))

print("\n--- Per-Type Detection Rate ---")
dr_cols = ["Target", "Source", "PairType", "Mode", "DR_spike", "DR_drop", "DR_coll"]
df_dr = df_summary[dr_cols].copy()
for col in ["DR_spike", "DR_drop", "DR_coll"]:
    df_dr[col] = df_dr[col].apply(lambda x: f"{x*100:.1f}%" if pd.notna(x) else "N/A")
print(df_dr.to_string(index=False))

# Paarvergleich
print("\n\n" + "=" * 130)
print("  PAARVERGLEICH: Aehnlich (+) vs. Unaehnlich (-)")
print("=" * 130)
for target_city in EXPERIMENT_PAIRS.keys():
    print(f"\n  -- {target_city} --")
    t_df = df_summary[df_summary["Target"] == target_city]
    oracle = t_df[t_df["Mode"] == "Oracle100"]
    to5 = t_df[t_df["Mode"] == "TargetOnly5"]
    zs_pos = t_df[(t_df["Mode"] == "ZeroShot") & (t_df["PairType"] == "+")]
    zs_neg = t_df[(t_df["Mode"] == "ZeroShot") & (t_df["PairType"] == "-")]
    ft_pos = t_df[(t_df["Mode"] == "FT5") & (t_df["PairType"] == "+")]
    ft_neg = t_df[(t_df["Mode"] == "FT5") & (t_df["PairType"] == "-")]
    for label, sub in [("Oracle 100%", oracle), ("Target-only 5%", to5),
                       ("ZS Strict (+)", zs_pos), ("ZS Strict (-)", zs_neg),
                       ("FT 5% (+)", ft_pos), ("FT 5% (-)", ft_neg)]:
        if len(sub) > 0:
            r = sub.iloc[0]
            src = r["Source"] if r["Source"] != "-" else ""
            pr = f"{r['PR-AUC']:.4f}" if pd.notna(r['PR-AUC']) else "N/A"
            f1 = f"{r['F1']:.4f}" if pd.notna(r['F1']) else "N/A"
            print(f"    {label:20s} {src:15s}  PR-AUC={pr}  F1={f1}")

# Type x Regime Matrices
print("\n\n--- Type x Regime Matrices (PR-AUC / F1) ---")
for r in all_results:
    rd = r.get("regime_detail")
    if rd is None or rd.empty:
        continue
    exp_name = r.get("experiment", f"{r['Target']}_{r['Mode']}_{r['Source']}")
    print(f"\n  [{exp_name}]")
    pivot_pr = rd.pivot(index="regime", columns="type", values="pr_auc")
    pivot_pr = pivot_pr.reindex(index=REGIMES, columns=ANOM_TYPES)
    print(pivot_pr.to_string(float_format="%.3f"))
    pivot_f1 = rd.pivot(index="regime", columns="type", values="f1")
    pivot_f1 = pivot_f1.reindex(index=REGIMES, columns=ANOM_TYPES)
    print("  (F1):")
    print(pivot_f1.to_string(float_format="%.3f"))


  V17c GESAMTVERGLEICH -- Cross-City Pair Comparison

--- Hauptmetriken ---
      Target         Source PairType        Mode  PR-AUC     F1  Precision  Recall  Threshold
    Mannheim              - baseline   Oracle100  0.6614 0.6584     0.7256  0.6026     4.7130
    Mannheim              - baseline TargetOnly5  0.6388 0.6431     0.7034  0.5923     4.4677
    Mannheim     Heidelberg        +    ZeroShot  0.6166 0.6210     0.6465  0.5975     4.2951
    Mannheim     Heidelberg        +         FT5  0.6628 0.6554     0.7468  0.5840     4.7873
    Mannheim        Marburg        -    ZeroShot  0.6202 0.5740     0.7376  0.4698     4.7022
    Mannheim        Marburg        -         FT5  0.6560 0.6522     0.7325  0.5878     4.6778
    Freiburg              - baseline   Oracle100  0.5419 0.5772     0.5801  0.5744     4.5541
    Freiburg              - baseline TargetOnly5  0.5869 0.5875     0.5862  0.5889     4.4002
    Freiburg     Heidelberg        +    ZeroShot  0.5797 0.6108     0.6353  0.

In [22]:
# 19 — Artefakte speichern
df_summary.to_csv(f"{RESULTS_DIR}/v17c_summary.csv", index=False)
print(f"Summary: {RESULTS_DIR}/v17c_summary.csv")

regime_parts = []
for r in all_results:
    rd = r.get("regime_detail")
    if rd is not None and not rd.empty:
        rd_copy = rd.copy()
        rd_copy["target"] = r["Target"]
        rd_copy["source"] = r["Source"]
        rd_copy["pair_type"] = r["PairType"]
        rd_copy["mode"] = r["Mode"]
        regime_parts.append(rd_copy)
if regime_parts:
    df_regime_all = pd.concat(regime_parts, ignore_index=True)
    df_regime_all.to_csv(f"{RESULTS_DIR}/v17c_regime_details.csv", index=False)
    print(f"Regime Details: {RESULTS_DIR}/v17c_regime_details.csv")

with open(f"{RESULTS_DIR}/v17c_config.json", "w") as f:
    cfg_dict = asdict(cfg)
    cfg_dict["experiment_pairs"] = {k: [(s, p) for s, p in v] for k, v in EXPERIMENT_PAIRS.items()}
    json.dump(cfg_dict, f, indent=2, default=str)
print(f"Config: {RESULTS_DIR}/v17c_config.json")

print("\n=== GESPEICHERTE ARTEFAKTE ===")
for fpath in sorted(glob.glob(f"{RESULTS_DIR}/**/*", recursive=True)):
    if os.path.isfile(fpath):
        size_mb = os.path.getsize(fpath) / 1e6
        print(f"  {os.path.relpath(fpath, RESULTS_DIR):50s} {size_mb:6.1f} MB")


Summary: /content/drive/MyDrive/BA_Colab/data/v17c_pair_comparison/v17c_summary.csv
Regime Details: /content/drive/MyDrive/BA_Colab/data/v17c_pair_comparison/v17c_regime_details.csv
Config: /content/drive/MyDrive/BA_Colab/data/v17c_pair_comparison/v17c_config.json

=== GESPEICHERTE ARTEFAKTE ===
  models/braunschweig_oracle_fc.keras                   0.2 MB
  models/freiburg_oracle_fc.keras                       0.2 MB
  models/kaiserslautern_source_fc.keras                 0.2 MB
  models/ludwigshafen_source_fc.keras                   0.2 MB
  models/mannheim_oracle_fc.keras                       0.2 MB
  models/marburg_source_fc.keras                        0.2 MB
  v17c_config.json                                      0.0 MB
  v17c_regime_details.csv                               0.0 MB
  v17c_summary.csv                                      0.0 MB
